<a href="https://colab.research.google.com/github/mdonbruce/AspNetDocs/blob/master/03_instructor_executed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3: Data Validation & Sanitization — Instructor (Executed)

**Objective:** Validate customer records and sanitize unsafe text fields.

**Dataset:** `customer_profiles_large.csv`

This notebook is a complete reference solution with outputs.

In [1]:
import pandas as pd, numpy as np

df = pd.read_csv('customer_profiles_large.csv')
df.head()

,customer_id,full_name,email,age,account_balance,zipcode,notes,risk_band
0,C100000,Emma Thompson,h0yafd89@example.com,21,5119.7,18346,NaN,medium
1,C100001,Amir Thompson,spaces @x.com,59,19071.67,54896,new_customer,medium
2,C100002,Ivy Davis,a@b,32,17415.47,59997,needs_followup,medium
3,C100003,Ethan Wilson,DROP TABLE users;@evil.com,28,NaN,49856,vip,medium
4,C100004,Kai Miller,DROP TABLE users;@evil.com,24,17862.87,97163,NaN,medium


In [2]:
def is_valid_email(email):
    if email is None:
        return False
    s = str(email).strip()
    if s.count('@') != 1 or ' ' in s:
        return False
    user, dom = s.split('@')
    return bool(user) and bool(dom) and ('.' in dom)

def is_valid_age(age):
    try:
        a = int(age)
        return 18 <= a <= 90
    except Exception:
        return False

def safe_balance(x):
    if x is None:
        return None
    s = str(x).strip().replace('$','').replace(',','')
    if s == '':
        return None
    try:
        v = float(s)
        if not np.isfinite(v) or v < -100 or v > 100000:
            return None
        return float(v)
    except Exception:
        return None

df['valid_email'] = df['email'].apply(is_valid_email)
df['valid_age'] = df['age'].apply(is_valid_age)
df['balance_num'] = df['account_balance'].apply(safe_balance)
df['notes_sanitized'] = df['notes'].astype(str).str.replace('<','').str.replace('>','')
df['is_record_valid'] = df['valid_email'] & df['valid_age'] & df['balance_num'].notna()
df[['valid_email','valid_age','balance_num','is_record_valid']].head()

,valid_email,valid_age,balance_num,is_record_valid
0,True,True,5119.70,True
1,False,True,19071.67,False
2,False,True,17415.47,False
3,False,True,NaN,False
4,False,True,17862.87,False


In [3]:
valid_pct = df['is_record_valid'].mean()*100
invalid = df[~df['is_record_valid']]
print(f'Valid records: {valid_pct:.2f}%')
{
 'invalid_email': int((~invalid['valid_email']).sum()),
 'invalid_age': int((~invalid['valid_age']).sum()),
 'invalid_balance': int((invalid['balance_num'].isna()).sum()),
}

Valid records: 69.59%


{'invalid_email': 1153, 'invalid_age': 980, 'invalid_balance': 611}